In [0]:
%python
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("volume", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

print(catalog)
print(schema)
print(volume)

In [0]:
create schema if not exists ${catalog}.${schema};

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_dataset_to_move_mapping (
    dataset_name string,
    s3_bucket_name string,
    wave string
);

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_recon_remediation (
    execution_id string,
    s3_bucket_name string,
    bucket_prefix string,
    dataset_name string,
    remediation_attempted_time timestamp,
    partitions_attempted int,
    partitions_remediated int,
    partitions_failed int,
    remediation_status string,
    error_msg string
);

In [0]:
%python
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

# Create directory structure
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{schema}/{volume}/dataset_mapping")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{schema}/{volume}/processed_datasets")

print("Volume and directories created successfully")
print(f"/Volumes/{catalog}/{schema}/{volume}/dataset_mapping")
print(f"/Volumes/{catalog}/{schema}/{volume}/processed_datasets")

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_table_inventory (
    execution_id string,
    s3_bucket_name string,
    bucket_prefix string,
    key string,
    partition_key string,
    extension string,
    size bigint,
    edp_run_id string,
    snapshot_date date,
    load_status string,
    last_modified_time timestamp
)
CLUSTER BY (s3_bucket_name, bucket_prefix);

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_dataset_mapping (
    dataset_name string,
    s3_bucket_name string,
    bucket_prefix string,
    dbx_catalog string,
    dbx_managed_table_schema string,
    datetime timestamp
);

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_table_candidates (
    execution_id string,
    s3_bucket_name string,
    bucket_prefix string,
    table_name string,
    total_file_count int,
    table_file_size_mb double,
    structured_file_count bigint,
    candidate_for_managed_table_creation string,
    managed_table_created string,
    schema_analysis_results string,
    compatible_schema_change boolean,
    earliest_migration_date timestamp,
    recon_job_run boolean,
    recon_status string,
    recon_execution_time timestamp,
    safe_migration_to timestamp,
    base_line_timestamp timestamp,
    compatible_schema_description array<string>,
    error_message string
);

In [0]:
CREATE TABLE if not exists ${catalog}.${schema}.ccbr_migration_recon_results (
    execution_id STRING,
    table_name STRING,
    recon_type STRING,
    exec_timestamp TIMESTAMP,
    exec_duration DOUBLE,
    status STRING,
    source_metrics STRING,
    target_metrics STRING,
    metrics_compared INT,
    metrics_matched INT,
    metrics_different INT,
    summary STRING,
    error_log STRING
)
USING delta;

In [0]:
%sql
create table if not exists ${catalog}.${schema}.ccbr_migration_parquet_schemas (
    execution_id string,
    file_path string,
    schema_json string,
    is_corrupted boolean,
    error_message string,
    bucket_prefix string,
    last_modified_time timestamp,
    run_timestamp timestamp,
    s3_bucket_name string
    -- snapshot_date string,
    -- edp_run_id string
)
CLUSTER BY (s3_bucket_name, bucket_prefix);

In [0]:
%sql
create table if not exists ${catalog}.${schema}.ccbr_migration_parquet_schema_history_table (
    execution_id string,
    file_path string,
    schema_json string,
    bucket_prefix string,
    schema_change boolean,
    schema_diff string,
    change_type string,
    is_corrupted boolean,
    processing_order int,
    last_modified_time timestamp,
    run_timestamp timestamp,
    s3_bucket_name string
)
CLUSTER BY (s3_bucket_name, bucket_prefix);

In [0]:
create table if not exists ${catalog}.${schema}.ccbr_migration_status (
    execution_id string,
    s3_bucket_name string,
    bucket_prefix string,
    table_name string,
    migration_status string,
    datetime timestamp
);

In [0]:
%sql
CREATE OR REPLACE VIEW ${catalog}.${schema}.v_ccbr_migration_bucket_volume_mapping AS
WITH tables_to_migrate AS
(
    SELECT DISTINCT
        CONCAT('s3://', s3_bucket_name, '/') AS storage_location,
        s3_bucket_name
        , bucket_prefix
    FROM ${catalog}.${schema}.ccbr_migration_dataset_mapping
)
SELECT
    DISTINCT t.s3_bucket_name AS s3_bucket_name,
    CONCAT('/Volumes/', v.volume_catalog, '/', v.volume_schema, '/', v.volume_name) AS volume_name
FROM
    system.information_schema.volumes v
    INNER JOIN tables_to_migrate t
        ON t.storage_location = v.storage_location
;

In [0]:
/*create table if not exists ${catalog}.${schema}.ccbr_migration_run_audit (
    execution_id string,
    `source` string,
    source_bucket_name string,
    source_bucket_prefix string,
    source_table_name string,
    `target` string,
    target_table_name string,
    job_id int,
    run_id int,
    run_by int,
    run_timestamp timestamp,
    run_status string,
    error_message string
)*/


In [0]:
/*create table if not exists ${catalog}.${schema}.ccbr_migration_source_to_target_mapping (
    execution_id string,
    source_s3_bucket string,
    source_bucket_prefix string,
    source_catalog_name string,
    source_schema_name string,
    source_table_name string,
    target_catalog_name string,
    target_schema_name string,
    target_table_name string,
    target_volume_name string
)*/

In [0]:
/*CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.ccbr_migration_audit_log (
    execution_id STRING,
    s3_bucket_name STRING,
    bucket_prefix STRING,
    source_table STRING,
    target_table STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    source_format STRING,
    source_record_count BIGINT,
    target_record_count BIGINT,
    file_count INT,
    status STRING,
    error_message STRING,
    user_id STRING,
    job_id STRING,
    cluster_id STRING
) USING DELTA*/